## **Dimension Table - Customers**

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.DimCustomers
AS
WITH Table AS (
  SELECT DISTINCT customer_id,customer_email, customer_name, customer_name_upper
  FROM datamodeling.silver.silver_table
) 
SELECT *, row_number() OVER (ORDER BY customer_id) AS CustomerKey
FROM Table

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM datamodeling.gold.DimCustomers

customer_id,customer_email,customer_name,customer_name_upper,CustomerKey
1,john.doe@example.com,John Doe,JOHN DOE,1
2,jane.smith@example.com,Jane Smith,JANE SMITH,2
3,alice.johnson@example.com,Alice Johnson,ALICE JOHNSON,3
6,david.white@example.com,David White,DAVID WHITE,4
7,eve.black@example.com,Eve Black,EVE BLACK,5


## **Dimension Table - Products**

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.DimProducts
AS
WITH Table_Pro AS (
  SELECT DISTINCT product_id,product_name,product_category
  FROM datamodeling.silver.silver_table
) 
SELECT *, row_number() OVER (ORDER BY product_id) AS ProductKey
FROM Table_Pro

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM datamodeling.gold.DimProducts

product_id,product_name,product_category,ProductKey
101,Product A,Category X,1
102,Product B,Category Y,2
103,Product C,Category X,3
106,Product F,Category X,4
107,Product G,Category Z,5


## **Dimension Table - Payments**

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.DimPayments
AS
WITH Table_Pay AS (
  SELECT DISTINCT payment_type
  FROM datamodeling.silver.silver_table
) 
SELECT *, row_number() OVER (ORDER BY payment_type) AS PaymentKey
FROM Table_Pay

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM datamodeling.gold.DimPayments

payment_type,PaymentKey
Apple Pay,1
Credit Card,2
PayPal,3


## **Dimension Table - Regions**

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.DimRegions
AS
WITH Table_Reg AS (
  SELECT DISTINCT country
  FROM datamodeling.silver.silver_table
) 
SELECT *, row_number() OVER (ORDER BY country) AS RegionKey
FROM Table_Reg

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM datamodeling.gold.DimRegions

country,RegionKey
Canada,1
USA,2


## **Dimension Table - Sales**

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.DimSales
AS
WITH Table_Sal AS (
  SELECT 
  order_id,
  order_date,
  customer_id,
  customer_name,
  customer_email,
  product_id,
  product_name,
  product_category,
  payment_type,
  country,
  last_updated
  FROM datamodeling.silver.silver_table
) 
SELECT *, row_number() OVER (ORDER BY order_id) AS SaleKey
FROM Table_Sal

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM datamodeling.gold.DimSales

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,payment_type,country,last_updated,SaleKey
1001,2022-01-01,1,John Doe,john.doe@example.com,101,Product A,Category X,Credit Card,USA,2022-01-01,1
1002,2022-01-02,2,Jane Smith,jane.smith@example.com,102,Product B,Category Y,PayPal,Canada,2022-01-02,2
1003,2022-01-03,3,Alice Johnson,alice.johnson@example.com,103,Product C,Category X,Apple Pay,USA,2022-01-03,3
1006,2022-01-06,6,David White,david.white@example.com,106,Product F,Category X,Apple Pay,USA,2022-01-06,4
1007,2022-01-07,7,Eve Black,eve.black@example.com,107,Product G,Category Z,PayPal,Canada,2022-01-07,5


## **Fact Table**

In [0]:
%sql
CREATE OR REPLACE TABLE datamodeling.gold.FactTable
AS
SELECT
S.SaleKey, C.CustomerKey, Pr.ProductKey, R.RegionKey, Pa.PaymentKey, F.quantity, F.unit_price
FROM datamodeling.silver.silver_table AS F
LEFT JOIN datamodeling.gold.DimSales AS S
  ON F.order_id = S.order_id
LEFT JOIN datamodeling.gold.DimCustomers AS C
  ON F.customer_id = C.customer_id
LEFT JOIN datamodeling.gold.DimProducts AS Pr
  ON F.product_id = Pr.product_id
LEFT JOIN datamodeling.gold.DimPayments AS Pa
  ON F.payment_type = Pa.payment_type
LEFT JOIN datamodeling.gold.DimRegions AS R
  ON F.country = R.country



num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM datamodeling.gold.FactTable

SaleKey,CustomerKey,ProductKey,RegionKey,PaymentKey,quantity,unit_price
1,1,1,2,2,2,10.00
2,2,2,1,3,1,15.00
3,3,3,2,1,3,20.00
4,4,4,2,1,4,35.00
5,5,5,1,3,2,40.00
